In [4]:
import kagglehub
kagglehub.login()

# export KAGGLE_API_TOKEN=KGAT_be96b99b82783874376c258e41c96de6

# Download latest version
path = kagglehub.competition_download('bt-4012-competition-2026')

print("Path to competition files:", path)

Path to competition files: C:\Users\regan\.cache\kagglehub\competitions\bt-4012-competition-2026


In [17]:
%pip install lightgbm scikit-learn

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ----------------------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import pandas as pd
import numpy as np

data_dir = r"C:\Users\regan\.cache\kagglehub\competitions\bt-4012-competition-2026"
print("Files:", os.listdir(data_dir))

# Read the core files (adjust filenames if slightly different in the directory listing)
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))
edges_df = pd.read_csv(os.path.join(data_dir, "txs_edgelist.csv")) # Contains txId1 -> txId2

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Edges shape: {edges_df.shape}")

Files: ['sample_submission.csv', 'test.csv', 'train.csv', 'txs_edgelist.csv']
Train shape: (141772, 168)
Test shape:  (15329, 168)
Edges shape: (234355, 2)


In [10]:
# Compute in-degree and out-degree across the entire transaction graph
in_degrees = edges_df.groupby("txId2").size().rename("in_degree")
out_degrees = edges_df.groupby("txId1").size().rename("out_degree")

def attach_graph_features(df):
    # Map degrees by transaction ID
    df = df.merge(in_degrees, left_on="txId", right_index=True, how="left")
    df = df.merge(out_degrees, left_on="txId", right_index=True, how="left")
    
    # Missing degrees mean the node has degree 0 in the observed edges
    df["in_degree"] = df["in_degree"].fillna(0).astype(int)
    df["out_degree"] = df["out_degree"].fillna(0).astype(int)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    df["degree_ratio"] = (df["in_degree"] + 1) / (df["out_degree"] + 1)
    
    return df

train_df = attach_graph_features(train_df)
test_df = attach_graph_features(test_df)

In [13]:
print("Train columns:", list(train_df.columns[:10]), "...")
print("Difference between train and test:", [c for c in train_df.columns if c not in test_df.columns])

Train columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8'] ...
Difference between train and test: ['label']


In [14]:
time_col = "time_step" if "time_step" in train_df.columns else "timestamp"
target_col = "target" if "target" in train_df.columns else "label"

# Filter out unlabeled rows (target == -1 or unknown) for supervised training
labeled_train = train_df[train_df[target_col].isin([0, 1])].copy()

# Temporal validation split: Train on earlier time steps, validate on the latest time steps
val_cutoff = labeled_train[time_col].quantile(0.8)

trn = labeled_train[labeled_train[time_col] < val_cutoff]
val = labeled_train[labeled_train[time_col] >= val_cutoff]

features = [col for col in train_df.columns if col not in ["txId", "index", target_col, time_col]]

X_tr, y_tr = trn[features], trn[target_col]
X_val, y_val = val[features], val[target_col]

print(f"Train samples: {len(X_tr)} (Time < {val_cutoff})")
print(f"Validation samples: {len(X_val)} (Time >= {val_cutoff})")

Train samples: 24923 (Time < 28.0)
Validation samples: 6312 (Time >= 28.0)


In [18]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Compute positive class weighting: count(licit) / count(illicit)
neg_count = (y_tr == 0).sum()
pos_count = (y_tr == 1).sum()
pos_weight = neg_count / max(pos_count, 1)

baseline_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=42
)

baseline_model.fit(X_tr, y_tr)

# Evaluate on temporal holdout
val_preds = baseline_model.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", roc_auc_score(y_val, val_preds))

[LightGBM] [Info] Number of positive: 2457, number of negative: 22466
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018887 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38416
[LightGBM] [Info] Number of data points in the train set: 24923, number of used features: 168
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.098584 -> initscore=-2.213062
[LightGBM] [Info] Start training from score -2.213062
Validation ROC-AUC: 0.996888569256375


In [ ]:
test_preds = baseline_model.predict_proba(test_df[features])[:, 1]

id_col = "index" if "index" in test_df.columns else "txId"

sub = pd.DataFrame({
    "index": test_df[id_col],
    "target": test_preds
})

sub.to_csv("submission_baseline.csv", index=False)
print("Submission saved! Top 5 rows:")
print(sub.head())